In [ ]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import numpy as np
from cop_ard_app.computation import create_client
from cop_ard_app.dem import import_dem, stack_dem
from cop_ard_app.io import Path, normpath, read, glob, delete
from cop_ard_app.area_projection import stack_geometry, stack_mask
from cop_ard_app.simulation import stack_simulation, stack_simulation_ml
from cop_ard_app.slc import (
    import_slc_stack,
    finalize_import,
    extract_orbits,
    extract_acquisition_dates,
)
from cop_ard_app.model import update_coreg
from cop_ard_app.etad import import_ETAD
from cop_ard_app.orbit import import_ORBIT
from cop_ard_app.coreg import stack_coregistration
from cop_ard_core.logger import logger
from cop_ard_app.utils import get_bbox

# Reference Image

In [ ]:
in_files = glob("/data/SENTINEL1/SURAT/SAFE/*SAFE")
path = "/data/SENTINEL1/SURAT/NOTEBOOK"
wdir_path_static = f"{path}/STATIC"
wdir_path_tmp = f"{path}/DYNAMIC"
slcs_path = f"{wdir_path_tmp}/slcs"

indices = None
chunks_sr = {"acquisition_date": 1, "polarization": 1, "azimuth_time": 2048, "slant_range_time": 2048}
chunks = {"x": 2048, "y": 2048}
chunk_length = max([v for v in chunks.values()])
spacing = {"x": 30, "y": 30}
recompute = True
multiburst = True
do_import = False if (recompute and multiburst) else True

In [ ]:
type, n_jobs, job_memory = ["LOCAL", 16, 0]
client = create_client(type, n_jobs, job_memory)
client

In [ ]:
slcs = import_slc_stack(
    [in_files[0]],
    slcs_path,
    deramp=True,
    multiburst=multiburst,
    noise_removal_flag=True,
    apply_invalid_mask_flag=True,
    indices=indices,
    calib_type="BetaZero",
    chunks=chunks_sr,
    do_import=do_import,
)

In [ ]:
slcs = xr.DataTree()
if multiburst:
    files = glob(f"{slcs_path}/bursts/*/orig/*")
    for f in files:
        name = f.strip(slcs_path)
        slcs[name] = read(f)
else:
    slcs = read(slcs_path)

In [ ]:
slcs, reference_date = finalize_import(slcs=slcs, working_directory=wdir_path_static, recompute=recompute)

In [ ]:
bbox = get_bbox(slcs)
multiburst = True if "bursts" in slcs else False
dem = import_dem(
    bbox=bbox,
    # dem_type=f"{wdir_path_tmp}/dem",
    working_dir=wdir_path_tmp,
    chunks=chunks,
)

In [ ]:
dem_tree = stack_dem(dem, slcs, multiburst, spacing, wdir_path_static, wdir_path_tmp, chunks, recompute=True)

In [ ]:
dem_tree = xr.DataTree()
for k, v in (slcs.bursts if multiburst else {"data": slcs}).items():
    path = f"bursts/{k}" if multiburst else "data"
    d = read(f"{wdir_path_tmp}/{path}/dem_resampled")
    if multiburst:
        dem_tree[path] = d
    else:
        dem_tree = xr.DataTree(d)

In [ ]:
orbits = extract_orbits(
    slcs,
    multiburst=multiburst,
    acquisition_date=[reference_date],
)

In [ ]:
simulation_tree = stack_simulation(
    dem_tree=dem_tree,
    orbits=orbits,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    chunks={**chunks, "acquisition_date": 1},
    master_image=True,
    recompute=recompute,
)

In [ ]:
simulation_ml_tree = stack_simulation_ml(
    simulation_tree=simulation_tree,
    reference_date=reference_date,
    multiburst=multiburst,
    working_directory=wdir_path_static,
    recompute=recompute,
    enable_dask=True,
)

In [ ]:
stack_geometry(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    working_directory_tmp=wdir_path_tmp,
    reference_date=reference_date,
    chunk_length=chunk_length,
    recompute=recompute,
    multiburst=multiburst,
    enable_dask=True,
)

In [ ]:
stack_mask(
    slcs,
    simulation_tree,
    simulation_ml_tree,
    working_directory_static=wdir_path_static,
    reference_date=reference_date,
    recompute=True,
    multiburst=multiburst,
    enable_dask=True,
)